In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
from dataclasses import dataclass

@dataclass
class Context:
    user_name: str
    is_premium: bool = False

In [7]:
from langchain.agents.middleware import wrap_model_call
from langchain.chat_models import init_chat_model

@wrap_model_call
def dynamic_model_selector(request, handler):
    user_name = request.runtime.context.user_name
    is_premium = request.runtime.context.is_premium

    if is_premium:
        model_name = "google_genai:gemini-3-flash-preview"
        print(f"{user_name} 사용자는 프리미엄 회원입니다. gemini-3-flash-preview를 호출합니다.")
    else:
        model_name = "google_genai:gemini-2.5-flash"
        print(f"{user_name} 사용자는 일반 회원입니다. gemini-2.5-flash를 호출합니다.")

    new_model = init_chat_model(model_name)
    new_request = request.override(model=new_model)
    
    return handler(new_request)

In [8]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[],
    middleware=[dynamic_model_selector],
    context_schema=Context
)

In [10]:
agent.invoke(
    {"messages": [{"role": "user", "content": "안녕하세요?"}]},
    context=Context(user_name="김일남", is_premium=True)
)

김일남 사용자는 프리미엄 회원입니다. gemini-3-flash-preview를 호출합니다.


{'messages': [HumanMessage(content='안녕하세요?', additional_kwargs={}, response_metadata={}, id='801cf2af-6e05-4198-9d47-3b64e75067cb'),
  AIMessage(content=[{'type': 'text', 'text': '안녕하세요! 반가워요. 무엇을 도와드릴까요? 무엇이든 물어보세요!', 'extras': {'signature': 'EtYFCtMFAb4+9vvM0lXQiT6/SnUtq82Cox6lyyZSZk/eS4YP1eH9vXLQgHDYI3SYc/wv+hw0nkOOMNj24NHZDAua23S4YJBZXkvR6tLArz61oLRX3f++V+MK2FxR90uDNNdgd0kal9TgXCjDV1O1s3IbInfKOlJMmLtG/uyaQY02i0JsihSaj+vN01MdC3iHTID4ie62gwE9QQqnYxYZNyUt9ny0+LsnaW0jHk6g9liK2Ne1qInSvMrmVwrzQP4n+aXpnpfsEzSTN03KYaNU4biB9MqGV0AfKJEN6bKhP0Z8pgCIgHLY5DWRL0Io+UA5NCxLusJwVxUliGvVR7DNhKbP8dsV+WbKdP3j5Sx7AZOQ5ggCb2W1tV+vQTUE/7yEVoiNMr1cXnSj6V7cq1xDEhJ2w1vQ82tChUqGxswjuYbmMLBddapSYdY/qKVtloeb+ZNs1cDy+shkAfmI+WiA3zwVXV7ziUeXNgekoQWVsF+uaaj3SZY+BJcmtyPVpfaKN100pFGkjVHYfk+KJsPGFRqXJrHwTGpnLojoTixauuf1MNvsj6+CdhMPHdfFkVDqnJFCet2FlC5IE3E/+0e4ApONT7YBmewdkePFiHScdpKQr48AXJP7ZYw9x2dUGee1XzKO9T2xOa5sZ6cC//Qz1PVUIeuiVKAtot2pOa5fndy7BPmwgAIw8k5XRdO/c4gbLkZSshuANTngOtLaWhkTOby6N1NGrqjzr63PbGR3OUR0hwlGOWd3

In [11]:
agent.invoke(
    {"messages": [{"role": "user", "content": "안녕하세요?"}]},
    context=Context(user_name="김일남")
)

김일남 사용자는 일반 회원입니다. gemini-2.5-flash를 호출합니다.


{'messages': [HumanMessage(content='안녕하세요?', additional_kwargs={}, response_metadata={}, id='55f0fe3e-f53e-49ca-85f5-d5f76e7ba1c0'),
  AIMessage(content='안녕하세요! 무엇을 도와드릴까요?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c36ff-3b23-7432-b5e1-d17bc330c43b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 3, 'output_tokens': 528, 'total_tokens': 531, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 520}})]}

---

In [22]:
from langchain.agents.middleware import wrap_model_call
from langchain.chat_models import init_chat_model

@wrap_model_call
def dynamic_model_selector(request, handler):
    last_msg = request.messages[-1].content if request.messages else ""
    msg_len = len(last_msg)

    if msg_len >= 10:
        model_name = "google_genai:gemini-3-flash-preview"
        print(model_name + "호출")
    elif msg_len >=  5:
        model_name = "google_genai:gemini-2.5-flash"
        print(model_name + "호출")
    else:
        model_name = "google_genai:gemini-2.5-flash-lite"
        print(model_name + "호출")

    new_model = init_chat_model(model_name)
    new_request = request.override(model=new_model)
    
    return handler(new_request)

In [24]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[],
    middleware=[dynamic_model_selector],
    context_schema=Context
)

In [25]:
agent.invoke(
    {"messages": [{"role": "user", "content": "안녕"}]},
    context=Context(user_name="김일남")
)

google_genai:gemini-2.5-flash-lite호출


{'messages': [HumanMessage(content='안녕', additional_kwargs={}, response_metadata={}, id='b01628d4-497f-4cdc-b992-fbb39adbf2c5'),
  AIMessage(content='안녕하세요! 무엇을 도와드릴까요?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c3709-db8d-7253-a415-1181db2889b2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 3, 'output_tokens': 8, 'total_tokens': 11, 'input_token_details': {'cache_read': 0}})]}

In [26]:
agent.invoke(
    {"messages": [{"role": "user", "content": "랭체인이 뭔가요?"}]},
    context=Context(user_name="김일남")
)

google_genai:gemini-2.5-flash호출


{'messages': [HumanMessage(content='랭체인이 뭔가요?', additional_kwargs={}, response_metadata={}, id='534223a3-eca6-478e-89e5-4afda59340d8'),
  AIMessage(content='랭체인(LangChain)은 **대규모 언어 모델(LLM)을 활용한 애플리케이션을 더 쉽고 강력하게 만들 수 있도록 도와주는 오픈소스 프레임워크 또는 라이브러리**입니다.\n\n쉽게 비유하자면, LLM을 \'똑똑한 뇌\'라고 생각했을 때, 랭체인은 그 뇌가 \'눈, 귀, 손발\'을 가지고 세상과 상호작용하고 복잡한 작업을 수행할 수 있도록 연결해주는 \'신경망\' 또는 \'도구상자\'와 같습니다.\n\n---\n\n### 왜 랭체인이 필요한가요?\n\nLLM 자체는 매우 강력하지만, 몇 가지 한계가 있습니다.\n\n1.  **실시간 정보 부족:** LLM은 학습된 데이터까지만 알고, 그 이후의 최신 정보나 실시간 웹 검색은 할 수 없습니다.\n2.  **행동 불가:** LLM은 글을 생성하거나 이해할 뿐, 외부 도구를 사용해 특정 행동(예: 웹 검색, 계산, 데이터베이스 조회, 이메일 전송)을 직접 수행할 수 없습니다.\n3.  **제한된 컨텍스트:** 한 번에 처리할 수 있는 정보의 양(토큰 수)에 제한이 있습니다. 긴 대화 기록이나 많은 문서를 한 번에 기억하기 어렵습니다.\n4.  **복잡한 작업 처리:** 여러 단계를 거쳐야 하는 복잡한 질문이나 요청을 한 번에 처리하기 어렵습니다.\n\n랭체인은 이러한 LLM의 한계를 극복하고, LLM을 활용하여 훨씬 더 유용하고 실용적인 애플리케이션을 만들 수 있도록 돕습니다.\n\n---\n\n### 랭체인의 주요 기능 (무엇을 할 수 있나요?)\n\n랭체인은 LLM을 다른 데이터 소스나 도구와 "연결(Chain)"하여 다양한 기능을 수행할 수 있게 해줍니다. 주요 기능은 다음과 같습니다.\n\n1.  **LLM 연동:** 다양한 LLM(

In [27]:
agent.invoke(
    {"messages": [{"role": "user", "content": "랭체인이 뭔가요? 초등학생도 알수 있게 쉽게 설명해 주세요."}]},
    context=Context(user_name="김일남")
)

google_genai:gemini-3-flash-preview호출


{'messages': [HumanMessage(content='랭체인이 뭔가요? 초등학생도 알수 있게 쉽게 설명해 주세요.', additional_kwargs={}, response_metadata={}, id='17faadd2-d8f5-4e8b-9f8b-e2cfe6d9fc08'),
  AIMessage(content=[{'type': 'text', 'text': '랭체인(LangChain)을 초등학생 친구들도 이해할 수 있게 아주 쉽게 설명해 줄게요!\n\n---\n\n### 1. 챗GPT는 아주 똑똑한 \'뇌\'예요\n먼저 챗GPT 같은 인공지능(AI)을 생각해 봐요. 챗GPT는 공부를 아주 많이 해서 아는 게 정말 많은 **\'똑똑한 뇌\'**와 같아요. 하지만 이 뇌는 컴퓨터 속에만 있어서, 혼자서는 책상을 정리하거나 최신 뉴스 신문을 가져오거나 일기를 대신 써주는 일을 하기는 조금 어려워요.\n\n### 2. 랭체인은 \'마법의 레고 블록\'이에요\n**랭체인**은 이 똑똑한 \'뇌(AI)\'에 **팔, 다리, 그리고 도구**를 달아주는 **\'마법의 연결 고리\'**라고 생각하면 돼요.\n\n이름을 잘 보면 **랭(Lang, 언어)** + **체인(Chain, 사슬)**이죠? AI가 말을 하는 능력(언어)을 다른 도구들과 사슬처럼 꽉 연결해 준다는 뜻이에요.\n\n### 3. 랭체인이 있으면 무엇을 할 수 있나요?\n\n예를 들어, 여러분이 "어제 우리 학교 급식 메뉴가 뭐였는지 알려주고, 그걸로 짧은 시를 써줘!"라고 시켰다고 해봐요.\n\n*   **AI 혼자서는:** "음... 저는 어제 학교 급식이 뭐였는지 몰라요."라고 할 거예요. (AI는 실시간 정보를 바로 알기 어렵거든요.)\n*   **랭체인을 쓰면:** \n    1.  **첫 번째 체인:** 학교 홈페이지에 가서 급식 메뉴를 **찾아봐요.** (검색 도구 연결)\n    2.  **두 번째 체인:** 찾은 메뉴 정보를 **AI에게 전달해요.**\n    3.  **세 번째 체인: